# Visualise Ancestry 

In [1]:
# visualize_ancestry.py

# Set the backend for interactive plotting
# This line must come BEFORE importing matplotlib.pyplot
import matplotlib
matplotlib.use('Qt5Agg') # Or 'TkAgg', 'WxAgg', 'MacOsX' depending on your system and preferences

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
import itertools
import re
import os
from typing import List, Tuple, Dict, Any, Optional, Literal # If you want to keep type hints

In [2]:
# --- Start of Plotting Functions to Copy ---

# (Ensure these imports are at the very top of your visualize_ancestry.py file)
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches
# from matplotlib.lines import Line2D # Needed for legend squares
# import os # Needed if you're saving plots

def get_unique_ancestries(locus_data_df: pd.DataFrame) -> List[Tuple[str, int]]:
    """
    Extracts all unique (pop_label, founder_id) tuples from the locus data DataFrame.
    """
    unique_ancestries = set()
    for _, row in locus_data_df.iterrows():
        anc_c1 = (row['ancestry_chromatid1_pop'], row['ancestry_chromatid1_founder_id'])
        anc_c2 = (row['ancestry_chromatid2_pop'], row['ancestry_chromatid2_founder_id'])
        if anc_c1 != (None, None): # Ensure not to add (None, None) if data is missing
            unique_ancestries.add(anc_c1)
        if anc_c2 != (None, None):
            unique_ancestries.add(anc_c2)
    return sorted(list(unique_ancestries)) # Sort for consistent color assignment

def generate_color_map_for_ancestries(unique_ancestries: List[Tuple[str, int]]) -> Dict[Tuple[str, int], str]:
    """
    Generates a consistent color map for unique ancestry (pop_label, founder_id) tuples.
    P_A founders get shades of blue, P_B founders get shades of red.
    """
    color_map = {}

    # Predefined base colors for P_A and P_B
    # Using more shades for potentially more distinct colors if many founders
    num_shades_a = max(10, len([anc for anc in unique_ancestries if anc[0] == 'P_A']))
    num_shades_b = max(10, len([anc for anc in unique_ancestries if anc[0] == 'P_B']))
    
    base_colors_A = plt.cm.Blues(np.linspace(0.4, 1.0, num_shades_a)) # Shades of blue
    base_colors_B = plt.cm.Reds(np.linspace(0.4, 1.0, num_shades_b))  # Shades of red

    # Separate founders by population and then sort by ID for consistent color assignment
    founders_A = sorted([anc for anc in unique_ancestries if anc[0] == 'P_A'], key=lambda x: x[1])
    founders_B = sorted([anc for anc in unique_ancestries if anc[0] == 'P_B'], key=lambda x: x[1])

    # Assign colors to P_A founders
    for i, anc in enumerate(founders_A):
        color_map[anc] = base_colors_A[i % len(base_colors_A)] # Cycle through shades if more founders than shades

    # Assign colors to P_B founders
    for i, anc in enumerate(founders_B):
        color_map[anc] = base_colors_B[i % len(base_colors_B)] # Cycle through shades

    # Handle any unexpected ancestry labels (e.g., if F1 or other intermediates accidentally get tagged as founder)
    other_founders = [anc for anc in unique_ancestries if anc[0] not in ['P_A', 'P_B']]
    for i, anc in enumerate(other_founders):
        # Use a distinguishable color like green for others, cycling if many
        other_colors = plt.cm.Greens(np.linspace(0.4, 1.0, len(other_founders) + 1))[i]
        color_map[anc] = other_colors

    return color_map


def plot_diploid_chromosome_ancestry_interactive(
    individual_id: int,
    diploid_chr_id: int,
    generation_label: str,
    locus_data_df: pd.DataFrame,
    marker_height: float = 0.8, # Height of the bar representing a locus
    spacing: float = 1.2,       # Spacing between chromatids
    save_filename: Optional[str] = None
):
    """
    Visualizes a specific diploid chromosome pair of an individual, showing allele values
    and color-coded ancestral origin for each locus. Features interactive hover to display
    detailed ancestry information.

    Args:
        individual_id (int): The ID of the individual to visualize.
        diploid_chr_id (int): The 1-based ID of the diploid chromosome pair (e.g., 1 for the first pair).
        generation_label (str): The generation label of the individual (e.g., 'F1', 'BC2A').
        locus_data_df (pd.DataFrame): The DataFrame containing locus-level genotype and ancestry data.
        marker_height (float): Height of the rectangular markers for each locus.
        spacing (float): Vertical spacing between the two chromatids.
        save_filename (str, optional): If provided, saves the plot to this filename.
    """

    # Filter data for the specific individual, chromosome, and generation
    target_data = locus_data_df[
        (locus_data_df['individual_id'] == individual_id) &
        (locus_data_df['diploid_chr_id'] == diploid_chr_id) &
        (locus_data_df['generation'] == generation_label)
    ].sort_values(by='locus_position').reset_index(drop=True)

    if target_data.empty:
        print(f"No data found for Individual ID {individual_id}, Chromosome {diploid_chr_id} in generation {generation_label}.")
        return

    num_loci = len(target_data)
    if num_loci == 0:
        print("No loci data available for plotting.")
        return

    # Get unique ancestries and generate a consistent color map
    # Pass full DF to ensure all ancestries are mapped correctly for consistent colors across plots
    unique_ancestries = get_unique_ancestries(locus_data_df)
    ancestry_colors = generate_color_map_for_ancestries(unique_ancestries)

    # --- Setup Plot ---
    # Adjust figure width dynamically based on number of loci for better readability
    fig_width = max(12, num_loci * 0.15) # Minimum 12 inches, then 0.15 inches per locus
    fig, ax = plt.subplots(figsize=(fig_width, 4))

    ax.set_title(f"Individual {individual_id}, Chromosome {diploid_chr_id} ({generation_label})\n"
                 "Ancestral Origin & Alleles with Hover Info", fontsize=14)
    ax.set_xlabel("Locus Position", fontsize=12)
    ax.set_ylabel("Chromatid", fontsize=12)
    ax.set_yticks([-spacing/2, spacing/2]) # Position labels for chromatids
    ax.set_yticklabels(['Chromatid 2', 'Chromatid 1']) # More intuitive labels
    
    # Dynamically set x-ticks to avoid clutter for many loci
    tick_interval = max(1, num_loci // 20) # At most 20 ticks
    ax.set_xticks(np.arange(0, num_loci, tick_interval))
    ax.set_xticklabels(np.arange(0, num_loci, tick_interval)) # Ensure labels match positions

    ax.set_xlim(-0.5, num_loci - 0.5)
    ax.set_ylim(-spacing, spacing) # Adjust y-limits to fit both chromatids and labels

    ax.set_aspect('auto', adjustable='box') # Let Matplotlib adjust aspect for better fit
    ax.axis('off') # Hide axes lines, we'll draw our own for clarity


    # --- Draw Chromosome Lines and Loci Rectangles ---
    # Draw central lines for chromatids
    ax.plot([-0.5, num_loci - 0.5], [spacing/2, spacing/2], color='black', linewidth=1.5, zorder=1) # Chromatid 1 line
    ax.plot([-0.5, num_loci - 0.5], [-spacing/2, -spacing/2], color='black', linewidth=1.5, zorder=1) # Chromatid 2 line

    # Store patch objects and their data for hover interactivity
    patches_list_c1 = []
    patches_list_c2 = []

    # Lists to store hover data for each patch
    hover_data_c1 = []
    hover_data_c2 = []

    for i in range(num_loci):
        locus_pos = target_data.loc[i, 'locus_position']

        # Chromatid 1
        allele_c1 = target_data.loc[i, 'genotype_allele_A']
        anc_pop_c1 = target_data.loc[i, 'ancestry_chromatid1_pop']
        anc_founder_c1 = target_data.loc[i, 'ancestry_chromatid1_founder_id']
        ancestry_c1 = (anc_pop_c1, anc_founder_c1) if pd.notna(anc_pop_c1) else None # Handle NaN for None
        color_c1 = ancestry_colors.get(ancestry_c1, 'lightgray') # Default to lightgray if ancestry not found

        rect_c1 = patches.Rectangle(
            (locus_pos - 0.5, spacing/2 - marker_height/2), # (x, y) bottom-left corner
            1, marker_height, # width, height
            facecolor=color_c1, edgecolor='darkgray', linewidth=0.2, zorder=2
        )
        ax.add_patch(rect_c1)
        patches_list_c1.append(rect_c1)
        hover_data_c1.append({
            'locus': locus_pos,
            'allele': allele_c1,
            'anc_pop': anc_pop_c1,
            'anc_founder': anc_founder_c1,
            'chromatid': 'Chromatid 1'
        })

        # Display allele value (optional, can be removed if too cluttered for many loci)
        # Only display if there aren't too many loci, or adjust font size
        if num_loci < 100: # Heuristic threshold
            ax.text(locus_pos, spacing/2 + marker_height/2 + 0.1, str(int(allele_c1)), # Convert to int for cleaner display
                    ha='center', va='bottom', fontsize=max(4, 10 - num_loci/20), color='black', zorder=3) # Adjust font size

        # Chromatid 2
        allele_c2 = target_data.loc[i, 'genotype_allele_B']
        anc_pop_c2 = target_data.loc[i, 'ancestry_chromatid2_pop']
        anc_founder_c2 = target_data.loc[i, 'ancestry_chromatid2_founder_id']
        ancestry_c2 = (anc_pop_c2, anc_founder_c2) if pd.notna(anc_pop_c2) else None
        color_c2 = ancestry_colors.get(ancestry_c2, 'lightgray')

        rect_c2 = patches.Rectangle(
            (locus_pos - 0.5, -spacing/2 - marker_height/2), # (x, y) bottom-left corner
            1, marker_height, # width, height
            facecolor=color_c2, edgecolor='darkgray', linewidth=0.2, zorder=2
        )
        ax.add_patch(rect_c2)
        patches_list_c2.append(rect_c2)
        hover_data_c2.append({
            'locus': locus_pos,
            'allele': allele_c2,
            'anc_pop': anc_pop_c2,
            'anc_founder': anc_founder_c2,
            'chromatid': 'Chromatid 2'
        })

        # Display allele value
        if num_loci < 100: # Heuristic threshold
            ax.text(locus_pos, -spacing/2 - marker_height/2 - 0.1, str(int(allele_c2)), # Convert to int
                    ha='center', va='top', fontsize=max(4, 10 - num_loci/20), color='black', zorder=3)


    # --- Interactive Hover Setup ---
    annot = ax.annotate("", xy=(0, 0), xytext=(15, 15), textcoords="offset points",
                        bbox=dict(boxstyle="round,pad=0.5", fc="yellow", ec="k", lw=1),
                        arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=0.3"))
    annot.set_visible(False)

    def update_annot(event_x, event_y, locus_idx, chromatid_type):
        if chromatid_type == 'C1':
            data = hover_data_c1[locus_idx]
            rect = patches_list_c1[locus_idx]
        else: # 'C2'
            data = hover_data_c2[locus_idx]
            rect = patches_list_c2[locus_idx]

        # Ensure correct coordinates for the annotation pointer
        x_center = rect.get_x() + rect.get_width() / 2
        y_center = rect.get_y() + rect.get_height() / 2

        annot.xy = (x_center, y_center)

        text = (f"Locus: {data['locus']}\n"
                f"Chromatid: {data['chromatid']}\n"
                f"Allele: {int(data['allele'])}\n" # Cast to int for cleaner display
                f"Origin Pop: {data['anc_pop'] if pd.notna(data['anc_pop']) else 'N/A'}\n"
                f"Founder ID: {int(data['anc_founder']) if pd.notna(data['anc_founder']) else 'N/A'}") # Cast to int for cleaner display

        annot.set_text(text)
        annot.get_bbox_patch().set_facecolor(rect.get_facecolor())
        annot.get_bbox_patch().set_alpha(0.8)

    def hover(event):
        vis = annot.get_visible()
        if event.inaxes == ax:
            found = False
            for i in range(num_loci):
                # Check Chromatid 1
                rect_c1 = patches_list_c1[i]
                contains_c1, _ = rect_c1.contains(event)
                if contains_c1:
                    update_annot(event.xdata, event.ydata, i, 'C1')
                    annot.set_visible(True)
                    fig.canvas.draw_idle()
                    found = True
                    break

                # Check Chromatid 2
                rect_c2 = patches_list_c2[i]
                contains_c2, _ = rect_c2.contains(event)
                if contains_c2:
                    update_annot(event.xdata, event.ydata, i, 'C2')
                    annot.set_visible(True)
                    fig.canvas.draw_idle()
                    found = True
                    break

            if not found:
                if vis:
                    annot.set_visible(False)
                    fig.canvas.draw_idle()
        else:
            if vis:
                annot.set_visible(False)
                fig.canvas.draw_idle()

    fig.canvas.mpl_connect("motion_notify_event", hover)

    # --- Add Legend for Ancestry Colors ---
    legend_elements = []
    # Create legend handles for P_A and P_B populations with distinct founder IDs
    # Sort unique_ancestries for consistent legend order
    sorted_unique_ancestries = sorted(unique_ancestries, key=lambda x: (x[0] if x[0] is not None else '', x[1] if x[1] is not None else float('inf')))

    for anc_tuple in sorted_unique_ancestries:
        pop_label = anc_tuple[0]
        founder_id = anc_tuple[1]
        color = ancestry_colors.get(anc_tuple, 'lightgray') # Get color from map

        if pop_label is not None and founder_id is not None:
             legend_elements.append(Line2D([0], [0], marker='s', color='w',
                                        label=f"{pop_label} (ID: {int(founder_id)})", # Cast founder_id to int for display
                                        markerfacecolor=color, markersize=10))
        elif pop_label is None and founder_id is None:
             # Handle cases where ancestry is None (e.g., if any loci are uninitialized or error)
             legend_elements.append(Line2D([0], [0], marker='s', color='w',
                                        label="N/A (Missing Ancestry)",
                                        markerfacecolor=color, markersize=10))


    if legend_elements:
        ax.legend(handles=legend_elements, title="Ancestral Origin",
                  loc='upper right', bbox_to_anchor=(1.05, 1), # Adjusted bbox_to_anchor for better spacing
                  borderaxespad=0., frameon=False, fontsize=9, title_fontsize=10)


    plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout to make space for the legend

    if save_filename:
        plt.savefig(save_filename, bbox_inches='tight', dpi=300)
        print(f"Chromosome plot saved to: {save_filename}")

    plt.show()

# --- End of Plotting Functions to Copy ---

In [3]:
# Define the base directory where your output data is stored
# IMPORTANT: YOU MUST CHANGE THIS PATH to where your CSVs are saved!
base_output_directory = r"C:\Users\sophi\Jupyter_projects\Hybrid_Code\output_data"
dataframes_directory = os.path.join(base_output_directory, "dataframes")
plots_directory = os.path.join(base_output_directory, "plots")

# Create the plots directory if it doesn't exist
os.makedirs(plots_directory, exist_ok=True)

locus_level_csv_path = os.path.join(dataframes_directory, "locus_level_genotypes_with_ancestry.csv")
# If you also want to plot HI/HET, you'll need the other data, e.g., if you saved `all_generations_data`
# all_generations_data_json_path = os.path.join(dataframes_directory, "all_generations_hi_het_data.json")


# Load the locus_level_df
try:
    locus_level_df = pd.read_csv(locus_level_csv_path)
    print(f"Successfully loaded {locus_level_csv_path}: {locus_level_df.shape[0]} rows.")
except FileNotFoundError:
    print(f"Error: Data file not found at {locus_level_csv_path}.")
    print("Please ensure your simulation script has been run and saved the data to this location.")
    exit() # Exit the script if the primary data isn't found
except Exception as e:
    print(f"An error occurred while loading the locus data: {e}")
    exit()

# If you have other dataframes or objects (like all_generations_data for HI/HET plot), load them here too.
# For example, if you saved all_generations_data as a JSON:
# try:
#     with open(all_generations_data_json_path, 'r') as f:
#         all_generations_data = json.load(f)
#     print(f"Successfully loaded {all_generations_data_json_path}.")
# except FileNotFoundError:
#     print(f"Warning: HI/HET data file not found at {all_generations_data_json_path}. HI/HET plots may not work.")
#     all_generations_data = None # Set to None or handle as appropriate
# except Exception as e:
#     print(f"An error occurred while loading HI/HET data: {e}")
#     all_generations_data = None

Successfully loaded C:\Users\sophi\Jupyter_projects\Hybrid_Code\output_data\dataframes\locus_level_genotypes_with_ancestry.csv: 1240000 rows.


In [4]:
if __name__ == "__main__":
    print("\n--- Starting Ancestry Visualization Script ---")

    # --- Example 1: List IDs in a specific generation ---
    generation_to_check = 'F1' # Change this to any generation you want to inspect
    print(f"\n--- Individual IDs in Generation '{generation_to_check}' ---")
    ids_in_generation = locus_level_df.loc[
        locus_level_df['generation'] == generation_to_check,
        'individual_id'
    ].unique().tolist()

    if ids_in_generation:
        print(f"Total individuals in '{generation_to_check}': {len(ids_in_generation)}")
        print("First 10 IDs (if available):", ids_in_generation[:10])
        print("Last 10 IDs (if available):", ids_in_generation[-10:])
    else:
        print(f"No individuals found for generation '{generation_to_check}'.")

    # --- Example 2: Plot the first individual of a chosen generation ---
    if ids_in_generation:
        chosen_gen_for_plot = generation_to_check # Or change to 'BC5A', etc.
        first_individual_id = ids_in_generation[0] # Get the very first ID
        chromosome_to_plot = 1 # Change to 2 if you want the second chromosome pair

        plot_save_path_first_ind = os.path.join(
            plots_directory,
            f"first_ind_{chosen_gen_for_plot}_chr_{chromosome_to_plot}_ancestry.png"
        )

        print(f"\n--- Plotting first individual (ID: {first_individual_id}) from generation '{chosen_gen_for_plot}', Chromosome {chromosome_to_plot} ---")
        plot_diploid_chromosome_ancestry_interactive(
            individual_id=first_individual_id,
            diploid_chr_id=chromosome_to_plot,
            generation_label=chosen_gen_for_plot,
            locus_data_df=locus_level_df,
            save_filename=plot_save_path_first_ind
        )
    else:
        print(f"\nSkipping plot for first individual: No IDs found for generation '{generation_to_check}'.")


    # --- Example 3: Plot a SPECIFIC individual by its ID ---
    # NOTE: Make sure this ID actually exists in your data for the specified generation!
    # You might get this ID from the list printed in Example 1, or from other analysis.
    specific_id_to_plot = 283 # <--- CHANGE THIS to an ID you know exists!
    specific_gen_for_id = 'F1' # <--- CHANGE THIS to the generation of that specific ID!
    specific_chr_to_plot = 2 # <--- CHANGE THIS to the chromosome pair (1 or 2)

    # Check if the specific individual exists in the DataFrame before attempting to plot
    if not locus_level_df[(locus_level_df['individual_id'] == specific_id_to_plot) &
                          (locus_level_df['generation'] == specific_gen_for_id) &
                          (locus_level_df['diploid_chr_id'] == specific_chr_to_plot)].empty:

        plot_save_path_specific_ind = os.path.join(
            plots_directory,
            f"specific_ind_{specific_id_to_plot}_gen_{specific_gen_for_id}_chr_{specific_chr_to_plot}_ancestry.png"
        )

        print(f"\n--- Plotting specific individual (ID: {specific_id_to_plot}) from generation '{specific_gen_for_id}', Chromosome {specific_chr_to_plot} ---")
        plot_diploid_chromosome_ancestry_interactive(
            individual_id=specific_id_to_plot,
            diploid_chr_id=specific_chr_to_plot,
            generation_label=specific_gen_for_id,
            locus_data_df=locus_level_df,
            save_filename=plot_save_path_specific_ind
        )
    else:
        print(f"\nSkipping plot for specific individual {specific_id_to_plot}: Data not found for this ID, generation, and chromosome combination.")

    # --- (Optional) Example 4: Plot Hybrid Index vs. Heterozygosity ---
    # This part requires the `plot_hi_het_triangle` function and its associated data (`all_generations_data`).
    # If you haven't implemented saving/loading `all_generations_data` in your simulation/viz scripts yet,
    # you can keep this commented out or work on adding that capability.
    # if 'plot_hi_het_triangle' in globals() and all_generations_data is not None:
    #     print("\n--- Plotting Hybrid Index vs. Heterozygosity (Means mode) ---")
    #     plot_hi_het_triangle(
    #         all_generations_data=all_generations_data,
    #         plot_mode='means',
    #         save_filename=os.path.join(plots_directory, "hi_het_means_plot.png")
    #     )
    # else:
    #     print("\nSkipping HI/HET plot: Function or data not available.")

    print("\n--- Visualization Script Finished ---")


--- Starting Ancestry Visualization Script ---

--- Individual IDs in Generation 'F1' ---
Total individuals in 'F1': 100
First 10 IDs (if available): [200, 201, 202, 203, 204, 205, 206, 207, 208, 209]
Last 10 IDs (if available): [290, 291, 292, 293, 294, 295, 296, 297, 298, 299]

--- Plotting first individual (ID: 200) from generation 'F1', Chromosome 1 ---


KeyboardInterrupt: 